# Part 1 · Build an API step by step, then read the real service

The product of this session is a **FastAPI service** in [`app/`](../app/): the Research Brief Agent behind an HTTP API,
plus a demo chat UI. Before reading it, this notebook builds a **small API** one step at a time, with the same code as
the slides, and calls it to print what each step returns. Then it shows where each step lives in the real service.

| Part | What it adds | Files of the real service |
| --- | --- | --- |
| **1** (this notebook) | a route, path / query / body data, validation (422), dependencies, routers, settings | `main.py`, `schemas.py`, `routers/`, `dependencies.py`, `core/config.py` |
| 2 | the agent's run streamed as Server-Sent Events | `agent/`, `services/streaming.py`, `routers/runs.py` |
| 3 | sign up, sign in (JWT), whose thread it is, human approval over HTTP | `core/security.py`, `routers/auth.py`, `services/threads.py` |
| 4 | one error format, health checks, the demo UI | `core/errors.py`, `routers/health.py`, `static/` |

Steps 1–6 need no server: `TestClient` sends real HTTP requests to an app inside the notebook. Steps 7–8 call the
**running service**, so start it now in a terminal in the session folder (keep it open: its log shows each request):
```bash
export UV_PROJECT_ENVIRONMENT=venvAE09        # Windows PowerShell: $env:UV_PROJECT_ENVIRONMENT="venvAE09"
uv run fastapi run --host 127.0.0.1
```

Docs: [First steps](https://fastapi.tiangolo.com/tutorial/first-steps/) ·
[Path parameters](https://fastapi.tiangolo.com/tutorial/path-params/) ·
[Query parameters](https://fastapi.tiangolo.com/tutorial/query-params/) · [Request body](https://fastapi.tiangolo.com/tutorial/body/) ·
[Dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/) ·
[Bigger Applications](https://fastapi.tiangolo.com/tutorial/bigger-applications/) · [Settings](https://fastapi.tiangolo.com/advanced/settings/)

In [1]:
import logging
import sys

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", stream=sys.stdout)  # log lines in the normal output
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
load_dotenv()

from fastapi.testclient import TestClient

from utils import PROJECT_DIR, check_server, client  # notebooks/utils.py


def call(app, method, url, **kwargs):
    """Send one request to an in-notebook app and print status + JSON body."""
    r = TestClient(app).request(method, url, **kwargs)
    print(f"{method} {url} -> {r.status_code} {r.json()}")
    return r

## Step 1. The smallest app: one route

`app = FastAPI()` creates the application. The **decorator** `@app.get("/health")` attaches an HTTP method (`GET`) and
a path (`/health`) to the function below it: together they are a **route** (FastAPI's docs call it a *path
operation*). Whatever the function returns, FastAPI converts to a JSON response with status `200`.

Saved as `main.py`, it runs with `fastapi dev main.py` (auto-reload while you edit). Here it runs in the notebook.

In [2]:
from fastapi import FastAPI

app = FastAPI()


@app.get("/health")
async def health():
    return {"status": "ok"}


call(app, "GET", "/health")
call(app, "GET", "/nothing-here");  # no route for this path: FastAPI answers 404

GET /health -> 200 {'status': 'ok'}
GET /nothing-here -> 404 {'detail': 'Not Found'}


### Methods: what the client wants to do

The method says what the client wants to do with the path: **GET** reads and changes nothing (no body), **POST**
sends data to create something or start a job (create a thread, run the agent), **PUT** replaces a resource,
**PATCH** changes part of it, **DELETE** removes it. FastAPI has one decorator per method (`@app.get`, `@app.post`,
`@app.put`, `@app.patch`, `@app.delete`). The same path with two methods is two routes; a method without a route
answers **405**.

In [3]:
THREADS = {"9fa034ac2308": {"thread_id": "9fa034ac2308", "title": "EV batteries"}}

app = FastAPI()


@app.get("/threads")
async def list_threads():
    return list(THREADS.values())


@app.post("/threads", status_code=201)
async def create_thread():
    THREADS["61b7c0d2e4aa"] = {"title": "New brief"}
    return THREADS["61b7c0d2e4aa"]


call(app, "GET", "/threads")
call(app, "POST", "/threads")
call(app, "GET", "/threads")
call(app, "DELETE", "/threads");

GET /threads -> 200 [{'thread_id': '9fa034ac2308', 'title': 'EV batteries'}]
POST /threads -> 201 {'title': 'New brief'}
GET /threads -> 200 [{'thread_id': '9fa034ac2308', 'title': 'EV batteries'}, {'title': 'New brief'}]
DELETE /threads -> 405 {'detail': 'Method Not Allowed'}


## Step 2. Data in the URL: path parameter and query parameter

A client puts data in three places: the **path** (`/threads/9fa034ac2308`), the **query** after `?`
(`/threads?limit=1`), and the **body** (step 4). FastAPI decides from the parameters of the function (name and type):
- a name that appears in the path (`{thread_id}`) is a **path parameter**;
- any other parameter of a simple type (`int`, `str`) is a **query parameter**; a default value makes it optional.

In [4]:
THREADS = {"9fa034ac2308": {"thread_id": "9fa034ac2308", "title": "EV batteries"},
           "61b7c0d2e4aa": {"thread_id": "61b7c0d2e4aa", "title": "Heat pumps"}}

app = FastAPI()


@app.get("/threads/{thread_id}")
async def get_thread(thread_id: str):
    return THREADS[thread_id]


@app.get("/threads")
async def list_threads(limit: int = 20):
    items = list(THREADS.values())
    return items[:limit]


call(app, "GET", "/threads/9fa034ac2308")
call(app, "GET", "/threads?limit=1")
call(app, "GET", "/threads");

GET /threads/9fa034ac2308 -> 200 {'thread_id': '9fa034ac2308', 'title': 'EV batteries'}
GET /threads?limit=1 -> 200 [{'thread_id': '9fa034ac2308', 'title': 'EV batteries'}]
GET /threads -> 200 [{'thread_id': '9fa034ac2308', 'title': 'EV batteries'}, {'thread_id': '61b7c0d2e4aa', 'title': 'Heat pumps'}]


## Step 3. Type hints: conversion and 422

A URL is only text. The **type hint** (`limit: int`) tells FastAPI to convert `"1"` to the integer `1`. When it
cannot, FastAPI answers **422** and the function never runs.

In [5]:
r = call(app, "GET", "/threads?limit=abc")

GET /threads?limit=abc -> 422 {'detail': [{'type': 'int_parsing', 'loc': ['query', 'limit'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}


The 422 body says **where** (`loc`: in the query, parameter `limit`) and **what** (`msg`), so the client can fix its
request. (`get_thread` with an unknown id would crash with `KeyError` → 500; step 5 fixes that.)

## Step 4. Data in the body: a Pydantic model

A `POST` sends JSON in the body. Its shape is a **Pydantic model** (a class that inherits `BaseModel`): field names,
types, limits. A parameter whose type is a model is read from the body. `extra="forbid"` refuses fields that are not
declared, such as a `user_id` a client tries to slip in.

In [6]:
from pydantic import BaseModel, ConfigDict, Field


class RunRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    message: str = Field(min_length=1, max_length=4000)


@app.post("/threads/{thread_id}/runs")
async def create_run(thread_id: str, body: RunRequest):
    return {"thread_id": thread_id, "message": body.message}


call(app, "POST", "/threads/9fa034ac2308/runs", json={"message": "Brief on heat pumps below 0°C."})
for bad in ({"message": ""}, {"message": "hi", "user_id": "alice"}, {"text": "hi"}):
    r = TestClient(app).post("/threads/9fa034ac2308/runs", json=bad)
    print(f"\n{bad} -> {r.status_code}", [(e["loc"], e["msg"]) for e in r.json()["detail"]])

POST /threads/9fa034ac2308/runs -> 200 {'thread_id': '9fa034ac2308', 'message': 'Brief on heat pumps below 0°C.'}

{'message': ''} -> 422 [(['body', 'message'], 'String should have at least 1 character')]

{'message': 'hi', 'user_id': 'alice'} -> 422 [(['body', 'user_id'], 'Extra inputs are not permitted')]

{'text': 'hi'} -> 422 [(['body', 'message'], 'Field required'), (['body', 'text'], 'Extra inputs are not permitted')]


## Step 5. Dependencies: a check shared by many routes

`Depends(f)` asks FastAPI to call `f` **before** the route and pass its result as the parameter. `f` reads path, query
and body parameters like a route does, and can depend on another function. When `f` raises `HTTPException`, the route
does not run. Here `owned_thread` looks the thread up once for every route that has a `thread_id`, and answers 404
instead of crashing.

In [7]:
from fastapi import Depends, HTTPException, Request

app = FastAPI()
app.state.store = THREADS  # objects created once at startup live on app.state


def get_store(request: Request):
    return request.app.state.store


async def owned_thread(thread_id: str, store=Depends(get_store)):
    thread = store.get(thread_id)
    if thread is None:
        raise HTTPException(404, "Thread not found")
    return thread


@app.get("/threads/{thread_id}")
async def get_thread(thread=Depends(owned_thread)):
    return thread


call(app, "GET", "/threads/9fa034ac2308")
call(app, "GET", "/threads/000000000000");

GET /threads/9fa034ac2308 -> 200 {'thread_id': '9fa034ac2308', 'title': 'EV batteries'}
GET /threads/000000000000 -> 404 {'detail': 'Thread not found'}


In the real service `owned_thread` also takes the signed-in user, and looks the thread up **in that user's data only**
(Part 3).

### The API documents itself

From the routes, type hints and models, FastAPI builds an **OpenAPI** document (`/openapi.json`) and serves
**Swagger UI** at `/docs`, where each route can be tried from the browser.

In [8]:
spec = app.openapi()
for path, ops in spec["paths"].items():
    for method, op in ops.items():
        params = [f"{p['name']} (in {p['in']})" for p in op.get("parameters", [])]
        print(f"{method.upper():<6}{path:<26}{op['summary']:<12}{params}")

GET   /threads/{thread_id}      Get Thread  ['thread_id (in path)']


## Step 6. `async`, `await` and the event loop

An `async def` function is a **coroutine**: it can stop at each `await` and continue later from the same place.
`await` waits for I/O (a model call, a web request) and hands the **event loop** back: the loop runs all coroutines on
one thread, so while request 1 waits, request 2 runs. A blocking call (`time.sleep`) keeps the loop busy, and
request 2 must wait.

To see it, the cell starts this small app on port 8001 and sends two requests at the same time (two client threads,
like two browsers). `log` prints the seconds since the requests were sent.

In [9]:
import asyncio
import threading
import time

import httpx
import uvicorn

app = FastAPI()
T0 = time.perf_counter()


def log(msg):
    print(f"{time.perf_counter() - T0:.1f} s  {msg}")


@app.get("/wait")
async def wait(n: int):
    log(f"request {n} start")
    await asyncio.sleep(2)  # stands for a 2-second model call
    log(f"request {n} end")
    return {"request": n}


@app.get("/block")
async def block(n: int):
    log(f"request {n} start")
    time.sleep(2)  # a blocking call: the event loop cannot run anything else
    log(f"request {n} end")
    return {"request": n}


server = uvicorn.Server(uvicorn.Config(app, port=8001, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.05)

for path in ("/wait", "/block"):
    print(path)
    T0 = time.perf_counter()
    clients = [threading.Thread(target=httpx.get, args=(f"http://127.0.0.1:8001{path}?n={n}",)) for n in (1, 2)]
    for c in clients:
        c.start()
        time.sleep(0.05)
    for c in clients:
        c.join()
server.should_exit = True

/wait
0.0 s  request 1 start
0.1 s  request 2 start
2.0 s  request 1 end
2.1 s  request 2 end
/block
0.0 s  request 1 start
2.0 s  request 1 end
2.0 s  request 2 start
4.0 s  request 2 end


With `await`, request 2 starts at once and both end after about 2 s. With `time.sleep`, request 2 starts only when
request 1 ends: 4 s in total. That is why the routes of the service are `async def` and call only functions they can
`await` (`agent.astream`).

### `async def` or `def` in a route

An `async def` route runs **on the event loop**, the one thread that serves all requests; inside it, only call
functions you can `await` (`agent.astream()`, `httpx.AsyncClient`), or every other request waits. A plain `def` route
is run by FastAPI **in a threadpool** (a set of worker threads), so blocking code (`requests`, `time.sleep`) is safe
there. The agent mostly waits on the model and the web, so the service's routes are `async def`.

In [10]:
import threading

app = FastAPI()


@app.get("/async")
async def on_event_loop():
    return {"thread": threading.current_thread().name}


@app.get("/sync")
def in_threadpool():
    return {"thread": threading.current_thread().name}


call(app, "GET", "/async")
call(app, "GET", "/sync");

GET /async -> 200 {'thread': 'asyncio-portal-70dbbf62f3e0'}
GET /sync -> 200 {'thread': 'AnyIO worker thread'}


`asyncio-portal-…` is the thread where `TestClient` runs the event loop (under `fastapi run` it is the main thread);
`AnyIO worker thread` is a thread of the threadpool.

## Step 7. The real service: the same steps, split into files

One file holding every route, model and check grows fast. FastAPI's *Bigger Applications* guide splits it:
- each group of routes goes in a module under `routers/` with an **`APIRouter`** (`@router.get` instead of
  `@app.get`); its `prefix` is the common start of the paths, e.g. `/threads/{thread_id}` in `routers/runs.py`, and
  `main.py` adds each router with `app.include_router(...)`;
- models in `schemas.py`, shared dependencies in `dependencies.py`;
- this service adds `core/` (settings, security, errors), `services/` (logic that does not touch HTTP) and `agent/`.

A request goes one way: `routers` → `dependencies` → `services` → `agent`.

> **The files of the service** (open them in the editor):
>
> | File | What it holds |
> | --- | --- |
> | `app/main.py` | the `FastAPI` object: startup (`lifespan`), `include_router`, error handlers, the UI |
> | `app/routers/` | the routes, one `APIRouter` per group: `health.py`, `auth.py`, `threads.py`, `runs.py` |
> | `app/dependencies.py` | checks that run before a route: `owned_thread`, `idle_thread`, `interrupted_thread` |
> | `app/schemas.py` | Pydantic models of requests and responses (`RunRequest`, `Thread`, …) |
> | `app/services/` | logic without HTTP: `threads.py` (thread records), `streaming.py` (a run as SSE events) |
> | `app/agent/` | the Research Brief Agent: `graph.py`, `tools.py`, `prompts/` |
> | `app/core/` | `config.py` (settings), `security.py` (passwords, JWT), `errors.py` (one error format) |
> | `app/static/` | the demo chat UI |

`list_threads` of the real service is step 2 plus two dependencies: `CurrentUser` (who calls, Part 3) and `Store`
(step 5). `Query(ge=1, le=100)` adds limits to the query parameter.

> **Read in the editor:** [`app/routers/threads.py`](../app/routers/threads.py), `list_threads`, lines 24–31.

In [11]:
check_server()  # GET /health on the running service; fails fast with the start command
carol = client("carol")  # signs up / signs in a demo user (Part 3 explains how)
for url in ("/threads?limit=2", "/threads?limit=abc", "/threads?limit=500"):
    r = carol.get(url)
    print(f"GET {url:<20} -> {r.status_code}", r.json() if r.status_code == 200 else r.json()["errors"])

server: ok
GET /threads?limit=2     -> 200 [{'thread_id': '8443cc07c78e', 'title': 'Brief on how well air-source heat pumps work below 0°C.', 'created_at': '2026-11-08T19:43:44+00:00', 'status': 'interrupted'}, {'thread_id': '23f9bd529f5b', 'title': 'Brief on how well air-source heat pumps work below 0°C.', 'created_at': '2026-11-08T18:01:24+00:00', 'status': 'interrupted'}]
GET /threads?limit=abc   -> 422 [{'loc': ['query', 'limit'], 'msg': 'Input should be a valid integer, unable to parse string as an integer'}]
GET /threads?limit=500   -> 422 [{'loc': ['query', 'limit'], 'msg': 'Input should be less than or equal to 100'}]


The 422 has the same `loc` and `msg` as step 3. The body around it is the service's single error format (Part 4).

## Step 8. Settings and startup

`Settings` (pydantic-settings) reads environment variables and the session `.env` once, with types and defaults:
`openai_api_key` is required, so the app refuses to start without it, and keys are `SecretStr`, so they never appear
in logs. `get_settings()` is cached: every module gets the same object.

> **Read in the editor:** [`app/core/config.py`](../app/core/config.py), `Settings`, lines 13–23.

In [12]:
from app.core.config import Settings, get_settings

settings = get_settings()
print("model:          ", settings.model)
print("run_timeout_s:  ", settings.run_timeout_s)
print("openai_api_key: ", repr(settings.openai_api_key))  # SecretStr hides the value
print("jwt_secret:     ", repr(settings.jwt_secret), "·", settings.jwt_algorithm, "·",
      settings.access_token_minutes, "min tokens")

import os

os.environ["RUN_TIMEOUT_S"] = "60"  # an environment variable wins over .env
print("run_timeout_s with RUN_TIMEOUT_S=60:", Settings().run_timeout_s)
del os.environ["RUN_TIMEOUT_S"]

model:           openai:gpt-6-luna
run_timeout_s:   180
openai_api_key:  SecretStr('**********')
jwt_secret:      SecretStr('**********') · HS256 · 30 min tokens
run_timeout_s with RUN_TIMEOUT_S=60: 60


`main.py` builds the `FastAPI` object. `lifespan` runs once at startup (model, checkpointer, store, agent: the
`app.state` objects of step 5) and closes the databases at shutdown. The FastAPI CLI finds the app through
`[tool.fastapi] entrypoint = "app.main:app"` in `pyproject.toml`:

| Command | Use |
| --- | --- |
| `uv run fastapi dev` | development: auto-reload on every file change (a reload stops runs that are streaming) |
| `uv run fastapi run --host 127.0.0.1` | serving: no reload. One worker process, because the registry of running threads is in memory |

> **Read in the editor:** [`app/main.py`](../app/main.py), `lifespan`, lines 29–43.

In [13]:
live = client(None).get("/openapi.json").json()
print(live["info"]["title"], "·", len(live["paths"]), "paths. Open http://127.0.0.1:8000/docs in the browser:")
for path, ops in live["paths"].items():
    print("  ", ", ".join(m.upper() for m in ops), path)

Research Brief Agent API · 9 paths. Open http://127.0.0.1:8000/docs in the browser:
   GET /health
   GET /ready
   POST /auth/signup
   POST /auth/token
   GET /auth/me
   POST, GET /threads
   GET /threads/{thread_id}
   POST /threads/{thread_id}/runs
   POST /threads/{thread_id}/resume
